# RLHF & InstructGPT: Aligning Language Models with Human Feedback

## Learning Objectives
1. Understand the three-stage RLHF pipeline
2. Implement supervised fine-tuning (SFT) on instruction data
3. Build and train a reward model for preference prediction
4. Apply PPO (Proximal Policy Optimization) for RL-based alignment

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from typing import Tuple, Dict, List
import matplotlib.pyplot as plt

# Device setup
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Level 1: Basic Supervised Fine-Tuning

Stage 1 of RLHF: Fine-tune base model on high-quality expert demonstrations.

In [ ]:
def basic_supervised_fine_tuning(
    prompts: List[str],
    responses: List[str],
    num_epochs: int = 2
) -> Dict[str, float]:
    """
    Simple supervised fine-tuning on instruction-response pairs.
    
    Args:
        prompts: List of input prompts
        responses: List of gold-standard responses
        num_epochs: Number of training epochs
    
    Returns:
        Training history
    """
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for prompt, response in zip(prompts, responses):
            # Simulate forward pass (cross-entropy loss)
            # In practice: tokenize, embed, compute log probability
            text_length = len(response.split())
            baseline_loss = np.log(1000) / text_length  # Vocabulary size ~ 1000
            
            # Simulate improvement over epochs
            loss = baseline_loss * (1 - epoch * 0.15)  # Loss decreases
            epoch_loss += loss
        
        avg_loss = epoch_loss / len(prompts)
        losses.append(avg_loss)
        print(f'Epoch {epoch+1}/{num_epochs}, SFT Loss: {avg_loss:.4f}')
    
    return {'losses': losses}

# Test SFT
sft_data = [
    ('What is machine learning?', 'Machine learning is a subset of AI where systems learn from data.'),
    ('How does gradient descent work?', 'Gradient descent updates parameters by moving in direction of negative gradient.'),
    ('Explain neural networks', 'Neural networks are interconnected layers that learn complex patterns through backpropagation.')
]

prompts = [p for p, _ in sft_data]
responses = [r for _, r in sft_data]

history = basic_supervised_fine_tuning(prompts, responses, num_epochs=3)
print(f'\nFinal SFT Loss: {history["losses"][-1]:.4f}')

## Level 2: Advanced RLHF Pipeline

Full three-stage pipeline: SFT → Reward Model → PPO training.

In [ ]:
class RLHFPipeline:
    """
    Complete RLHF training pipeline with all three stages.
    """
    
    def __init__(self, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        self.sft_losses = []
        self.reward_losses = []
        self.ppo_metrics = []
    
    def stage1_sft(
        self,
        prompts: List[str],
        responses: List[str],
        num_epochs: int = 2
    ) -> float:
        """
        Stage 1: Supervised Fine-Tuning on expert demonstrations.
        
        Args:
            prompts: Training prompts
            responses: Expert responses
            num_epochs: Training epochs
        
        Returns:
            Final SFT loss
        """
        print('\n=== Stage 1: Supervised Fine-Tuning ===')
        
        for epoch in range(num_epochs):
            epoch_loss = 0
            for prompt, response in zip(prompts, responses):
                # Simulate cross-entropy loss
                vocab_size = 50000  # Realistic tokenizer vocab
                baseline = np.log(vocab_size) / len(response.split())
                loss = baseline * np.exp(-epoch * 0.3)  # Exponential decay
                epoch_loss += loss
            
            avg_loss = epoch_loss / len(prompts)
            self.sft_losses.append(avg_loss)
            print(f'  Epoch {epoch+1}/{num_epochs}: SFT Loss = {avg_loss:.4f}')
        
        return self.sft_losses[-1]
    
    def stage2_reward_model(
        self,
        preference_pairs: List[Dict],
        num_epochs: int = 2
    ) -> float:
        """
        Stage 2: Train reward model to predict human preferences.
        
        Args:
            preference_pairs: List of (prompt, chosen, rejected) tuples
            num_epochs: Training epochs
        
        Returns:
            Final reward model loss
        """
        print('\n=== Stage 2: Training Reward Model ===')
        
        for epoch in range(num_epochs):
            epoch_loss = 0
            for pair in preference_pairs:
                # Bradley-Terry loss: P(chosen > rejected | prompt)
                # Loss = -log(sigmoid(r_chosen - r_rejected))
                
                # Simulate reward model outputs
                r_chosen = np.random.randn() + 0.5  # Slightly positive
                r_rejected = np.random.randn() - 0.5  # Slightly negative
                
                # Bradley-Terry loss
                diff = r_chosen - r_rejected
                loss = -np.log(1 / (1 + np.exp(-diff)))  # Log sigmoid
                epoch_loss += loss
            
            avg_loss = epoch_loss / len(preference_pairs)
            self.reward_losses.append(avg_loss)
            print(f'  Epoch {epoch+1}/{num_epochs}: Reward Loss = {avg_loss:.4f}')
        
        return self.reward_losses[-1]
    
    def stage3_ppo(
        self,
        prompts: List[str],
        num_steps: int = 10,
        beta_kl: float = 0.1
    ) -> List[Dict]:
        """
        Stage 3: PPO optimization with learned reward model.
        
        Args:
            prompts: Test prompts for RL training
            num_steps: Number of PPO steps
            beta_kl: KL penalty coefficient
        
        Returns:
            List of training metrics
        """
        print('\n=== Stage 3: PPO Optimization ===')
        
        for step in range(num_steps):
            # Sample responses from current policy
            rewards = np.random.randn(len(prompts)) + 0.5
            
            # Simulate policy gradients
            policy_loss = np.random.randn() * 0.1
            kl_div = np.log(1 + step * 0.1)  # Slight divergence over time
            total_loss = policy_loss + beta_kl * kl_div
            
            metrics = {
                'step': step,
                'avg_reward': float(rewards.mean()),
                'policy_loss': float(policy_loss),
                'kl_divergence': float(kl_div),
                'total_loss': float(total_loss)
            }
            self.ppo_metrics.append(metrics)
            
            if (step + 1) % 5 == 0:
                print(f'  Step {step+1}/{num_steps}: Reward={metrics["avg_reward"]:.3f}, '
                      f'Loss={metrics["total_loss"]:.4f}, KL={metrics["kl_divergence"]:.4f}')
        
        return self.ppo_metrics
    
    def run_full_pipeline(
        self,
        sft_prompts: List[str],
        sft_responses: List[str],
        preference_pairs: List[Dict],
        rl_prompts: List[str]
    ) -> Dict:
        """
        Run complete RLHF pipeline.
        """
        print('Starting RLHF Training Pipeline')
        print('=' * 50)
        
        sft_loss = self.stage1_sft(sft_prompts, sft_responses, num_epochs=2)
        reward_loss = self.stage2_reward_model(preference_pairs, num_epochs=2)
        ppo_metrics = self.stage3_ppo(rl_prompts, num_steps=10, beta_kl=0.1)
        
        print('\n' + '=' * 50)
        print('RLHF Pipeline Complete')
        
        return {
            'sft_loss': sft_loss,
            'reward_loss': reward_loss,
            'ppo_metrics': ppo_metrics
        }

# Run full pipeline
pipeline = RLHFPipeline(device=device)

sft_data = [
    ('What is ML?', 'Machine learning learns from data.'),
    ('How does GD work?', 'Gradient descent optimizes by updating parameters.'),
    ('Neural nets?', 'Neural networks learn complex patterns.')
]

preference_data = [
    {'prompt': 'What is AI?', 'chosen': 'AI is broad field...', 'rejected': 'AI bad.'},
    {'prompt': 'ML basics?', 'chosen': 'ML learns from data...', 'rejected': 'IDK'},
]

result = pipeline.run_full_pipeline(
    sft_prompts=[p for p, _ in sft_data],
    sft_responses=[r for _, r in sft_data],
    preference_pairs=preference_data,
    rl_prompts=[p for p, _ in sft_data]
)

## Real-World Example 1: PPO Training with Advantage Estimation

Implement Proximal Policy Optimization with Generalized Advantage Estimation (GAE).

In [ ]:
class PPOTrainer:
    """
    Proximal Policy Optimization trainer for RLHF.
    """
    
    def __init__(
        self,
        beta_kl: float = 0.1,
        clip_epsilon: float = 0.2,
        learning_rate: float = 1e-4,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    ):
        """
        Initialize PPO trainer.
        
        Args:
            beta_kl: KL penalty strength
            clip_epsilon: PPO clipping parameter
            learning_rate: Optimizer learning rate
            device: Training device
        """
        self.beta_kl = beta_kl
        self.clip_epsilon = clip_epsilon
        self.learning_rate = learning_rate
        self.device = device
    
    def compute_gae(
        self,
        rewards: np.ndarray,
        values: np.ndarray,
        gamma: float = 0.99,
        lambda_: float = 0.95
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute Generalized Advantage Estimation.
        
        Args:
            rewards: Reward trajectory from environment
            values: Value function estimates
            gamma: Discount factor
            lambda_: GAE lambda parameter
        
        Returns:
            advantages: Computed advantages
            returns: Cumulative discounted returns
        """
        advantages = []
        gae = 0
        
        # Reverse iteration for advantage computation
        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0
            else:
                next_value = values[t + 1]
            
            delta = rewards[t] + gamma * next_value - values[t]
            gae = delta + gamma * lambda_ * gae
            advantages.insert(0, gae)
        
        advantages = np.array(advantages)
        returns = advantages + values
        
        return advantages, returns
    
    def compute_ppo_loss(
        self,
        log_probs_new: torch.Tensor,
        log_probs_old: torch.Tensor,
        advantages: torch.Tensor,
        kl_div: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute PPO clipped objective + KL penalty.
        
        Args:
            log_probs_new: Log probs from current policy
            log_probs_old: Log probs from old policy
            advantages: Computed advantages
            kl_div: KL divergence from reference
        
        Returns:
            PPO loss
        """
        # Importance sampling ratio
        ratio = torch.exp(log_probs_new - log_probs_old)
        
        # PPO clipped objective
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1 - self.clip_epsilon, 1 + self.clip_epsilon) * advantages
        ppo_loss = -torch.min(surr1, surr2).mean()
        
        # Total loss with KL penalty
        total_loss = ppo_loss + self.beta_kl * kl_div.mean()
        
        return total_loss
    
    def training_step(
        self,
        batch_size: int = 32
    ) -> Dict[str, float]:
        """
        Simulate a PPO training step.
        
        Args:
            batch_size: Batch size for training
        
        Returns:
            Metrics dictionary
        """
        # Simulate trajectory data
        rewards = np.random.randn(batch_size) + 0.3  # Positive rewards
        values = np.random.randn(batch_size) * 0.5
        
        # Compute advantages
        advantages, returns = self.compute_gae(rewards, values, gamma=0.99, lambda_=0.95)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Simulate log probabilities
        log_probs_old = torch.randn(batch_size, device=self.device) - 1.0
        log_probs_new = log_probs_old + torch.randn(batch_size, device=self.device) * 0.1
        kl_div = (log_probs_old - log_probs_new).abs()
        
        advantages_tensor = torch.tensor(advantages, dtype=torch.float32, device=self.device)
        
        # Compute loss
        loss = self.compute_ppo_loss(log_probs_new, log_probs_old, advantages_tensor, kl_div)
        
        # Metrics
        ratio = torch.exp(log_probs_new - log_probs_old).mean().item()
        
        return {
            'loss': loss.item(),
            'avg_reward': np.mean(rewards),
            'avg_advantage': np.mean(advantages),
            'kl_divergence': np.mean(kl_div.detach().cpu().numpy()),
            'importance_ratio': ratio
        }

# Run PPO training
ppo_trainer = PPOTrainer(beta_kl=0.1, clip_epsilon=0.2)

print('Running PPO Training Steps:')
for step in range(5):
    metrics = ppo_trainer.training_step(batch_size=32)
    print(f"Step {step+1}: Loss={metrics['loss']:.4f}, "
          f"Reward={metrics['avg_reward']:.3f}, "
          f"KL={metrics['kl_divergence']:.4f}")

## Real-World Example 2: Reward Model Evaluation and Validation

Analyze reward model quality through Bradley-Terry analysis and preference accuracy.

In [ ]:
class RewardModelValidator:
    """
    Validate reward model quality on preference data.
    """
    
    def __init__(self):
        self.predictions = []
        self.ground_truth = []
    
    def evaluate_on_preferences(
        self,
        prompts: List[str],
        chosen_responses: List[str],
        rejected_responses: List[str]
    ) -> Dict[str, float]:
        """
        Evaluate reward model accuracy on preference pairs.
        
        Args:
            prompts: Input prompts
            chosen_responses: Human-preferred responses
            rejected_responses: Dispreferred responses
        
        Returns:
            Evaluation metrics
        """
        correct = 0
        margin_scores = []  # Score margin between chosen and rejected
        
        for prompt, chosen, rejected in zip(prompts, chosen_responses, rejected_responses):
            # Simulate reward model scores
            # Good reward model: chosen > rejected
            r_chosen = len(chosen.split()) * 0.1 + np.random.randn() * 0.1
            r_rejected = len(rejected.split()) * 0.05 + np.random.randn() * 0.1
            
            # Check if model correctly ranks preferences
            if r_chosen > r_rejected:
                correct += 1
            
            margin = r_chosen - r_rejected
            margin_scores.append(margin)
        
        accuracy = correct / len(prompts)
        avg_margin = np.mean(margin_scores)
        margin_std = np.std(margin_scores)
        
        return {
            'accuracy': float(accuracy),
            'avg_margin': float(avg_margin),
            'margin_std': float(margin_std),
            'num_examples': len(prompts)
        }
    
    def analyze_calibration(
        self,
        reward_scores: np.ndarray,
        human_preference_labels: np.ndarray
    ) -> Dict[str, float]:
        """
        Analyze reward model calibration: do high scores correlate with preferences?
        
        Args:
            reward_scores: Predicted reward scores
            human_preference_labels: Human judgments (0 or 1)
        
        Returns:
            Calibration metrics
        """
        # Bin predictions and compute empirical frequencies
        bins = np.linspace(reward_scores.min(), reward_scores.max(), 5)
        empirical_probs = []
        bin_centers = []
        
        for i in range(len(bins) - 1):
            mask = (reward_scores >= bins[i]) & (reward_scores < bins[i+1])
            if mask.sum() > 0:
                empirical_prob = human_preference_labels[mask].mean()
                empirical_probs.append(empirical_prob)
                bin_centers.append((bins[i] + bins[i+1]) / 2)
        
        # Expected probability should equal sigmoid(score)
        expected_probs = 1.0 / (1 + np.exp(-np.array(bin_centers)))
        
        # Calibration error
        calibration_error = np.mean(np.abs(np.array(empirical_probs) - expected_probs))
        
        return {
            'calibration_error': float(calibration_error),
            'num_bins': len(empirical_probs)
        }

# Test validator
validator = RewardModelValidator()

test_prompts = ['What is AI?'] * 10
test_chosen = ['AI is intelligence...'] * 10
test_rejected = ['AI is bad.'] * 10

metrics = validator.evaluate_on_preferences(test_prompts, test_chosen, test_rejected)

print('Reward Model Evaluation:')
for key, value in metrics.items():
    if isinstance(value, float):
        print(f'  {key}: {value:.4f}')
    else:
        print(f'  {key}: {value}')

# Test calibration
reward_scores = np.random.randn(100) * 2
labels = (reward_scores > 0).astype(int)
calib = validator.analyze_calibration(reward_scores, labels)
print(f'\nCalibration error: {calib["calibration_error"]:.4f}')

## Key Takeaways

**The Three-Stage RLHF Pipeline:**
1. **SFT (Supervised Fine-Tuning)**: Train on expert demonstrations to establish baseline instruction-following
2. **Reward Modeling**: Train model to predict human preferences (Bradley-Terry model)
3. **PPO (Policy Optimization)**: Optimize language model to maximize learned reward while staying close to SFT

**RLHF vs Modern Alternatives:**
| Aspect | RLHF | DPO | Constitutional AI |
|--------|------|-----|------------------|
| Stages | 3 (SFT, Reward, RL) | 1 (direct loss) | 1 (principle-based) |
| Speed | Slow (72 hours) | Fast (12 hours) | Very fast (hours) |
| Stability | Medium (RL instability) | High (supervised) | High (supervised) |
| Flexibility | High (multi-objective) | Medium (single-objective) | Medium (principle-based) |

**Key challenges in RLHF:**
- Reward hacking: Model maximizes learned reward without achieving true alignment
  → Solution: Use ensemble reward models, periodic human evaluation
- Distribution shift: Reward model trained on SFT outputs, RL explores beyond
  → Solution: Maintain moderate KL penalty, collect fresh preference data
- Data quality: Preferences must be consistent
  → Solution: High inter-rater agreement, expert annotators

**Production patterns:**
- Use moderate beta_kl (0.1-0.5) to balance reward vs stability
- Validate reward model on held-out preferences
- Monitor for reward hacking through human evaluation
- Combine RLHF (for helpfulness) with Constitutional AI (for safety rules)

## Exercises

1. **Analyze reward divergence**: What happens if the KL penalty (beta_kl) is too low? Implement an experiment showing divergence over time.

2. **Investigate reward hacking**: Design a scenario where the reward model is exploited (e.g., model always outputs confident wrong answers if those score high).

3. **Compare PPO variants**: Implement PPO with and without clipping. How does clipping affect convergence stability?

4. **Build preference data**: Create a system that automatically generates preference pairs (e.g., using heuristics). What are failure modes?